In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [ ]:
# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [ ]:
IHK = pd.read_csv(PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv')

# To geodataframe based on latitude and longitude columns
IHK_gdf = gpd.GeoDataFrame(
    IHK,
    geometry=gpd.points_from_xy(IHK['longitude'], IHK['latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:3035')

In [ ]:
# from e.g. '1 - 3 Beschäftigte' to mean(1,3) = 2
def parse_employees_range(range_str):
    if pd.isna(range_str):
        return np.nan
    
    if range_str == 'unbekannt':
        return np.nan
    
    try:
        parts = range_str.split('-')
        if len(parts) == 2:
            low = int(parts[0].strip())
            high = int(parts[1].strip().split()[0])  # Remove 'Beschäftigte'
            return (low + high) / 2
        else:
            return int(range_str.strip().split()[0])  # Single value case
    except Exception as e:
        print(f"Error parsing '{range_str}': {e}")
        return np.nan

IHK_gdf['empl'] = IHK_gdf['employees_range'].apply(parse_employees_range)

In [ ]:
# Assign to grid cells - preserving grid cell ID in IHK_gdf for filtering
IHK_grid = gpd.sjoin(IHK_gdf, grid, how='left', predicate='within')

# Aggregate employment by grid cell
empl_by_cell = IHK_grid.groupby('index')['empl'].sum().reset_index()
empl_by_cell.columns = ['index', 'empl']

# Merge employment data back to grid
grid = grid.reset_index(drop=True).merge(empl_by_cell, left_index=True, right_on='index', how='left')
grid['empl'] = grid['empl'].fillna(0)

In [ ]:
# Load the gebaeude and stadtstruktur data
gebaeude = gpd.read_file(PATH_RAW / 'gebaeude.gpkg')
stadtstruktur = gpd.read_file(PATH_RAW / 'stadtstruktur.gpkg')
zentren_fma = gpd.read_file(PATH_RAW / 'zentren.gpkg', layer="zentren_fma")
zentren_zh = gpd.read_file(PATH_RAW / 'zentren.gpkg', layer="zentren_zh")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

# ── 0. Filter ────────────────────────────────────────────────────────────────
filtered_gebaeude = (
    gebaeude[gebaeude['bezeich'] == 'AX_Gebaeude']
    .copy()
    .reset_index(drop=True)      # clean 0..N positional index
)
filtered_gebaeude['_bpos'] = filtered_gebaeude.index   # stable key survives all merges

# Stadtstruktur: materialise its integer position as an explicit column
stadtstruktur_indexed = stadtstruktur.copy().reset_index(drop=True)
stadtstruktur_indexed['_spos'] = stadtstruktur_indexed.index

# ── 1. Single intersects sjoin (how='left' → all buildings kept) ──────────────
# 'intersects' strictly subsumes 'within', so no two-pass logic needed.
sjoin = gpd.sjoin(
    filtered_gebaeude[['_bpos', 'geometry']],
    stadtstruktur_indexed[['_spos', 'geometry']],
    how='left',
    predicate='intersects',
)
# sjoin index == filtered_gebaeude positional index; index_right column dropped
# by predicate='intersects' — geopandas stores right-side positional as _spos via
# explicit column, not index_right, so we're safe.

# ── 2. Tie-break multi-matches by maximum intersection area ───────────────────
# (buildings straddling a stadtstruktur boundary intersect 2+ polygons; keep
#  the one with the largest overlap — most semantically correct assignment)
matched_mask = sjoin['_spos'].notna()

if matched_mask.any():
    matched = sjoin[matched_mask].copy()

    # Vectorised intersection area (Shapely 2.x — no Python loop)
    bldg_geoms_arr = filtered_gebaeude.geometry.values[matched['_bpos'].values]
    ss_geoms_arr   = stadtstruktur_indexed.geometry.values[matched['_spos'].astype(int).values]
    matched['_isect_area'] = shapely.area(
        shapely.intersection(bldg_geoms_arr, ss_geoms_arr)
    )

    # Best stadtstruktur per building = largest overlap
    best = (
        matched
        .sort_values('_isect_area', ascending=False)
        .drop_duplicates(subset='_bpos', keep='first')   # one row per building
        [['_bpos', '_spos', '_isect_area']]
    )
else:
    best = pd.DataFrame(columns=['_bpos', '_spos', '_isect_area'])

# ── 3. Attach match back to ALL buildings (left join preserves no-match rows) ──
gebaeude_with_ss_id = filtered_gebaeude.merge(best, on='_bpos', how='left')

# Classify match type for diagnostics
gebaeude_with_ss_id['match_type'] = np.where(
    gebaeude_with_ss_id['_spos'].notna(), 'intersects', 'no_match'
)

# ── 4. Bring in stadtstruktur attributes (no geometry column in right side) ────
ss_attrs = stadtstruktur_indexed.drop(columns=['geometry'])

gebaeude_stadtstruktur = gpd.GeoDataFrame(
    gebaeude_with_ss_id.merge(ss_attrs, on='_spos', how='left'),
    geometry='geometry',          # geometry column came from filtered_gebaeude,
    crs=filtered_gebaeude.crs,    # survived the left merge intact
)

# ── 5. Rename for downstream compatibility ─────────────────────────────────────
gebaeude_stadtstruktur = gebaeude_stadtstruktur.rename(columns={'_spos': 'stadtstruktur_id'})

# ── 6. Diagnostics ────────────────────────────────────────────────────────────
n_intersects = (gebaeude_stadtstruktur['match_type'] == 'intersects').sum()
n_no_match   = (gebaeude_stadtstruktur['match_type'] == 'no_match').sum()
n_total      = len(gebaeude_stadtstruktur)

print(f"Total AX_Gebaeude:       {n_total:>7,}")
print(f"Intersects match:        {n_intersects:>7,}  ({100*n_intersects/n_total:.1f}%)")
print(f"No stadtstruktur match:  {n_no_match:>7,}  ({100*n_no_match/n_total:.1f}%)")
print(f"Geometry NaN check:      {gebaeude_stadtstruktur.geometry.isna().sum()} NaN geometries")
print(f"CRS:                     {gebaeude_stadtstruktur.crs}")

In [ ]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
import matplotlib.pyplot as plt
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
    df_to_plot.plot(ax=ax, color='red', markersize=10)
    plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
    plt.show()


In [ ]:
# Unique count of bezgfk values for buildings with no stadtstruktur match
df_to_plot['bezgfk'].value_counts()

# If bezgfk == "Tiefgarage" -> aog = 0, bezgfk == "Garage" -> aog = 0, "Gebäude zum Parken" -> aog = 0, "Umformer" -> aog = 1, "Schutzbunker" -> aog = 0
# "Heizwerk" -> aog = 1, "Gebäude zur Elektrizitätsversorgung" -> aog = 1, "Parkdeck" -> aog = 0, "Speichergebäude" -> aog = 1, "Gebäude für Vorratshaltung" -> aog = 1
# "Pumpstation" -> aog = 0, "Lagerhalle, Lagerschuppen, Lagerhaus" -> aog = 1, "Gebäude zum Sportplatz" -> aog = 1, "Sport-, Turnhalle" -> aog = 1, "Gebäude für Sportzwecke" -> aog = 1,
# Wasserbehälter, Pumpwerk (nicht für Wasserversorgung) -> aog = 0
# Gebäude zur Wasserversorgung, Gebäude zur Abwasserbeseitigung, Hallenbad, Schuppen, Gartenhaus, Wasserwerk, Gebäude zur Abfallbehandlung, Gebäude für Land- und Forstwirtschaft, Bootshaus, Tierschauhaus, Müllbunker, Parkhaus -> aog = 1

# Create a mapping of bezgfk to aog based on the above logic
bezgfk_to_aog = {
    "Tiefgarage": 0,
    "Garage": 0,
    "Gebäude zum Parken": 0,
    "Umformer": 1,
    "Schutzbunker": 0,
    "Heizwerk": 1,
    "Gebäude zur Elektrizitätsversorgung": 1,
    "Parkdeck": 0,
    "Speichergebäude": 1,
    "Gebäude für Vorratshaltung": 1,
    "Pumpstation": 0,
    "Lagerhalle, Lagerschuppen, Lagerhaus": 1,
    "Gebäude zum Sportplatz": 1,
    "Sport-, Turnhalle": 1,
    "Gebäude für Sportzwecke": 1,
    "Wasserbehälter": 0,
    "Pumpwerk (nicht für Wasserversorgung)": 0,
    "Gebäude zur Wasserversorgung": 1,
    "Gebäude zur Abwasserbeseitigung": 1,
    "Hallenbad": 1,
    "Schuppen": 1,
    "Gartenhaus": 1,
    "Wasserwerk": 1,
    "Gebäude zur Abfallbehandlung": 1,
    "Gebäude für Land- und Forstwirtschaft": 1,
    "Bootshaus": 1,
    "Tierschauhaus": 1,
    "Müllbunker": 1,
    "Parkhaus": 1
}

# Apply the mapping to fill aog values for buildings with no stadtstruktur match
gebaeude_stadtstruktur['aog'] = gebaeude_stadtstruktur.apply(
    lambda row: bezgfk_to_aog.get(row['bezgfk'], np.nan) if pd.isna(row['aog']) else row['aog'],
    axis=1
)

In [ ]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
    df_to_plot.plot(ax=ax, color='red')
    plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
    plt.show()


In [ ]:
gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]['bezgfk'].value_counts()

old_gebaeude_stadtstruktur = gebaeude_stadtstruktur.copy()

# -> Just remove these from the gebaude_stadtstruktur

gebaeude_stadtstruktur = gebaeude_stadtstruktur[~gebaeude_stadtstruktur['aog'].isna()]

# HOH to bool
gebaeude_stadtstruktur['hoh'] = gebaeude_stadtstruktur['hoh'].apply(lambda x: False if x == 'false' or x == '' else True)

In [ ]:
from hotelling.spatial.gebaeude_capacity import (
    get_efficiency_factor,
    compute_usable_floor_area,
    compute_employee_hard_cap,
)

gebaeude_stadtstruktur["efficiency"] = gebaeude_stadtstruktur.apply(
    lambda r: get_efficiency_factor(r["gfk"], bool(r["hoh"])), axis=1
)
gebaeude_stadtstruktur["usable_area_m2"] = gebaeude_stadtstruktur.apply(
    lambda r: compute_usable_floor_area(
        r['shape_area'],
        r["aog"],
        r["gfk"],
        bool(r["hoh"]),
    ), axis=1
)

gebaeude_stadtstruktur["employee_hard_cap"] = gebaeude_stadtstruktur.apply(
    lambda r: compute_employee_hard_cap(
        r["shape_area"],
        r["aog"],
        r["gfk"],
        bool(r["hoh"]),
    ), axis=1
)

# Tags that have employees_hard_cap = 0: Wohnnutzung, Wochenendhaus- und kleingartenähnliche Nutzung, Kleingartenanlage, Park / Grünfläche, Friedhof, Wald, Grünland, Brachfläche, Mischbestand aus Wiesen, Gebüschen und Bäume, Ackerland, Brachfläche, wiesenartiger Vegetationsbestand, Gewässer, Brachfläche, vegetationsfrei

zero_employee_tags = [
    "Wohnnutzung",
    "Wochenendhaus- und kleingartenähnliche Nutzung",
    "Kleingartenanlage",
    "Park / Grünfläche",
    "Friedhof",
    "Wald",
    "Grünland",
    "Brachfläche, Mischbestand aus Wiesen, Gebüschen und Bäume",
    "Ackerland",
    "Brachfläche, wiesenartiger Vegetationsbestand",
    "Gewässer",
    "Brachfläche, vegetationsfrei"
]

def is_zero_employee(row):
    if str(row['nutzung']) in zero_employee_tags:
        return True
    return False

for id, row in gebaeude_stadtstruktur.iterrows():
    if is_zero_employee(row):
        gebaeude_stadtstruktur.at[id, 'employee_hard_cap'] = 0

gebaeude_stadtstruktur["effective_employees"] = gebaeude_stadtstruktur["employee_hard_cap"].apply(lambda x: np.round(x) if x != np.inf else 0)

In [ ]:
# take unique pairs of long and lat and count the employees for each pair on IHK_gdf

IHK_gdf['lon_lat'] = IHK_gdf.apply(lambda row: (row['longitude'], row['latitude']), axis=1)
empl_by_location = IHK_gdf.groupby('lon_lat')['empl'].sum().reset_index()
empl_by_location.columns = ['lon_lat', 'empl']

empl_by_location['count'] = IHK_gdf.groupby('lon_lat')['empl'].count().values

In [ ]:
import numpy as np
import geopandas as gpd
from shapely import STRtree         # Shapely 2.x vectorised API
from joblib import Parallel, delayed
import os

# ── 1. Single CRS transform (not inside any loop) ────────────────────────────
points_gdf = gpd.GeoDataFrame(
    empl_by_location.copy(),
    geometry=gpd.points_from_xy(
        empl_by_location['lon_lat'].map(lambda t: t[0]),
        empl_by_location['lon_lat'].map(lambda t: t[1]),
    ),
    crs='EPSG:4326',
).to_crs(gebaeude_stadtstruktur.crs)

# ── 2. Stable integer-indexed building array ──────────────────────────────────
buildings = gebaeude_stadtstruktur.reset_index(drop=True)   # positional index = tree index
bldg_geoms = buildings.geometry.values                       # numpy array of shapely geoms
bldg_ids   = buildings['id'].values
bldg_areas = buildings.geometry.area.values

# ── 3. STRtree: O(n log m) nearest-polygon query ─────────────────────────────
tree = STRtree(bldg_geoms)
point_geoms = points_gdf.geometry.values   # numpy array

# `nearest` returns one index per query point; ties broken by tree insertion order
nearest_idxs = tree.nearest(point_geoms)   # shape (N,), dtype int64

# ── 4. Vectorised distance computation (all CPUs via joblib) ─────────────────
N_JOBS   = min(os.cpu_count(),1)
CHUNK    = max(1, len(point_geoms) // N_JOBS)

def _dist_chunk(pts, bldgs):
    """Compute element-wise distances for a chunk."""
    return np.array([p.distance(b) for p, b in zip(pts, bldgs)])

chunks_pts  = [point_geoms[i:i+CHUNK]  for i in range(0, len(point_geoms),  CHUNK)]
chunks_bldg = [bldg_geoms[nearest_idxs[i:i+CHUNK]] for i in range(0, len(nearest_idxs), CHUNK)]

dist_chunks = Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(_dist_chunk)(p, b) for p, b in zip(chunks_pts, chunks_bldg)
)
distances = np.concatenate(dist_chunks)

# ── 5. Tie detection and area-based resolution ────────────────────────────────
# A tie exists where another building has the same (within float tol) distance.
# We resolve by smallest area among all candidates at that distance.

def _resolve_ties(pt_idx, point, dist_to_nearest):
    """Return building positional index, resolving ties by smallest area."""
    tol = 1e-2   # metres; tweak if buildings share exact boundary edges
    candidate_idxs = tree.query(point.buffer(dist_to_nearest + tol))   # candidates within reach
    candidate_dists = np.array([point.distance(bldg_geoms[i]) for i in candidate_idxs])
    tied = candidate_idxs[np.abs(candidate_dists - dist_to_nearest) < tol]
    if len(tied) <= 1:
        return nearest_idxs[pt_idx]   # no tie
    # Break tie: smallest building area → most specific physical structure
    return tied[np.argmin(bldg_areas[tied])]

# Only invoke tie resolution for zero-distance points (containment ambiguity)
# and any explicit ties flagged above. Filter to points with d=0 first.
zero_mask = distances < 1e-2

resolved_idxs = nearest_idxs.copy()
tie_count = 0

for i in np.where(zero_mask)[0]:
    ri = _resolve_ties(i, point_geoms[i], distances[i])
    if ri != nearest_idxs[i]:
        tie_count += 1
    resolved_idxs[i] = ri

print(f"Ties resolved via smallest-area rule: {tie_count}")

# ── 6. Write results back ─────────────────────────────────────────────────────
empl_by_location['building_id']          = bldg_ids[resolved_idxs]
empl_by_location['distance_to_building'] = distances

# Diagnostic: how many points fell inside a building vs. outside?
print(f"Points inside a building (d=0): {(distances < 1e-2).sum()}")
print(f"Points outside nearest building: {(distances >= 1e-2).sum()}")
print(f"Max distance to nearest building: {distances.max():.1f} m")

In [ ]:
empl_by_location.sort_values('distance_to_building', ascending=False)

In [ ]:
# plot buildings from gebaeude_stadtstruktur and points from empl_by_location colored by distance to building
import matplotlib.pyplot as plt

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    gebaeude_stadtstruktur.plot(ax=ax, color='lightgrey', edgecolor='black', alpha=0.05)
    #old_gebaeude_stadtstruktur.plot(ax=ax, color='none', edgecolor='blue', alpha=0.1)
    #stadtstruktur.plot(ax=ax, color='none', edgecolor='red', alpha=0.1)
    # Color points by distance to building
    points_gdf = gpd.GeoDataFrame(empl_by_location.copy(), geometry=gpd.points_from_xy(
        empl_by_location['lon_lat'].map(lambda t: t[0]),
        empl_by_location['lon_lat'].map(lambda t: t[1]),
    ), crs='EPSG:4326').to_crs(gebaeude_stadtstruktur.crs)

    points_gdf = points_gdf[points_gdf['distance_to_building'] > 100]  # Filter to points within 100m for better visualization

    points_gdf.plot(ax=ax, column='distance_to_building', cmap='viridis', markersize=10, legend=True)
    plt.title('Distance from IHK points to nearest building')
    plt.show()

In [ ]:
empl_by_building = empl_by_location.groupby('building_id')['empl'].sum()

gebaeude_stadtstruktur = gebaeude_stadtstruktur.merge(
    empl_by_building.rename('empl'),
    left_on='id',
    right_index=True,
    how='left'
)

def row_min(a,b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    else:
        return min(a,b)
    
v_row_min = np.vectorize(lambda a,b: row_min(a,b))

gebaeude_stadtstruktur['approx_empl'] = v_row_min(gebaeude_stadtstruktur['empl'].values, gebaeude_stadtstruktur['employee_hard_cap'].values)


In [ ]:
gebaeude_stadtstruktur['centroid'] = gebaeude_stadtstruktur.geometry.centroid

gebaeude_stadtstruktur.sort_values('approx_empl', ascending=False)[['id', 'approx_empl', 'employee_hard_cap', 'centroid']].head(20)

In [ ]:
gebaeude_centroid = gebaeude_stadtstruktur.copy()
gebaeude_centroid['geometry'] = gebaeude_centroid.geometry.centroid

# Handle NaN values and normalization safely
max_empl = gebaeude_centroid['approx_empl'].replace([np.inf, -np.inf], np.nan).max()
gebaeude_centroid['approx_empl_norm'] = gebaeude_centroid['approx_empl'] / max_empl if max_empl > 0 else 0
gebaeude_centroid['approx_empl_norm'] = gebaeude_centroid['approx_empl_norm'].fillna(0)

if GENERATE_PLOTS:
    # Plot buildings colored by approx_empl with alpha dependent on approx_empl_norm
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Filter to non-zero employment buildings
    to_plot = gebaeude_centroid[gebaeude_centroid['approx_empl_norm'] != 0].copy()

    # Extract coordinates
    x = to_plot.geometry.x.values
    y = to_plot.geometry.y.values
    c = to_plot['approx_empl_norm'].values

    # Make alpha dependent on approx_empl_norm (scale to [0.1, 0.9] for visibility)
    v_min = np.vectorize(lambda x, y: min(x, y))
    alphas = v_min(0.05 + 0.1 * np.exp(c), 1)

    # Plot with variable alpha
    cmap = plt.get_cmap('viridis')
    scatter = ax.scatter(x, y, c=c, cmap=cmap, s=5, alpha=alphas, vmin=0, vmax=1)
    plt.colorbar(scatter, ax=ax, label='Normalized Employment')

    berlin.to_crs(gebaeude_stadtstruktur.crs).plot(ax=ax, color='none', edgecolor='black', linewidth=1)

    plt.title('Buildings colored and glowing by approximate employment')
    plt.show()


In [ ]:
empl_scatter = gebaeude_centroid[['geometry', 'approx_empl', 'employee_hard_cap']]
empl_scatter['approx_empl'] = empl_scatter['approx_empl'].apply(lambda x: int(np.round(x)) if not np.isnan(x) else 0)

In [ ]:
empl_scatter.to_crs('EPSG:4326').sort_values('approx_empl', ascending=False).head(30)

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from aabpl_wrapper import detect_employment_clusters

result, summary = detect_employment_clusters(empl_scatter, k_percentile = 99.5, min_empl = 10, radius_m = 500, )

In [ ]:
result

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fix, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Add OSM basemap
    result_web = result.to_crs(berlin.crs)
    # Plot clusters and boundary
    result_web.plot(ax=ax, column='cluster_id', markersize=5, legend=True, alpha=1)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=result_web.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)

    plt.title('Detected employment clusters with OSM background')
    plt.show()

In [ ]:
# Output full building gdf
gebaeude_stadtstruktur.to_parquet(PATH_PROCESSED / 'gebaeude_stadtstruktur.parquet', index=False)

# Output the cluster data
result.to_parquet(PATH_PROCESSED / 'employment_clusters.parquet', index=False)

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_02_city_data.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")